<div align="center">

# Assignment 9

</div>

# Autoencoders and Variational Autoencoders

Deadline: May 26, 2026

# 🎯 Objective

The goal of this assignment is to:

* understand the basic idea of an **autoencoder** as an encoder-decoder model,
* compare a simple linear autoencoder with **PCA**,
* train a **denoising autoencoder** on Fashion-MNIST,
* practice the first mathematical ideas behind **Variational Autoencoders**,
* calculate simple KL divergence terms used in VAE training,
* implement the **reparameterization trick**,
* optionally try inference with a pretrained VAE decoder/encoder from Hugging Face Diffusers.

This assignment is connected to [Lecture 11, May 19](/doc/6df8323e-5acb-41a3-bc52-294cdff8e80a) : Autoencoders and Variational Autoencoders.


---

# 📌 General Requirements

* Use **PyTorch** and **TorchVision**.
* Submit a **Jupyter Notebook (.ipynb)**.
* The notebook must:
  * run from top to bottom without errors,
  * include markdown explanations,
  * include plots where required,
  * set a random seed,
  * use clear variable names,
  * be reproducible.

You may use GPU if available, but the assignment should also be possible on CPU if you use small models and/or subsets of the data.


---

# ⚠ Restrictions

✅ Allowed:

* `torch.nn.Module`
* `torch.optim`
* `torch.utils.data.DataLoader`
* `torchvision.datasets.FashionMNIST`
* `torchvision.transforms`
* `matplotlib`
* `numpy`
* `sklearn.decomposition.PCA` only for the PCA comparison in Task 1
* Hugging Face `diffusers` only for the optional pretrained VAE inference part

❌ Not allowed:

* high-level training frameworks such as Lightning, fastai, or Trainer APIs,
* copying a full autoencoder/VAE implementation from the internet without explanation,
* using a pretrained model for Tasks 1 or 2,
* skipping the training loop.

You must write:

* the model class,
* the forward pass,
* the loss computation,
* the training loop,
* the evaluation / visualization code,
* the discussion of results.


---

# 📚 Recommended Reading

Before starting, review:

* [PyTorch `torch.nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
* [TorchVision Fashion-MNIST dataset](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html)
* [scikit-learn PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
* [Hugging Face Diffusers `AutoencoderKL`](https://huggingface.co/docs/diffusers/api/models/autoencoderkl)

Note: TorchVision provides many pretrained models for classification, detection, segmentation, video, and optical flow, but it does not provide a standard pretrained VAE model in the same way it provides pretrained ResNet or MobileNet models. For pretrained VAE inference, use Hugging Face Diffusers.


---

# Task 1 — Linear autoencoder and PCA on 3D data (3 pts)

In this task, you will train a very small autoencoder with architecture:

$$
3 \rightarrow 2 \rightarrow 3
$$

The goal is to compress 3-dimensional data into a 2-dimensional latent representation and reconstruct the original input.

A linear autoencoder trained with mean squared error learns a low-dimensional subspace similar to PCA. The exact coordinates in the latent space may be different, but the reconstruction quality and the learned subspace should be comparable.


---

## Dataset

Generate a simple 3D dataset with strong correlation between coordinates.

Example:

```python
import torch

torch.manual_seed(42)

n = 1000

u = torch.randn(n, 1)
v = torch.randn(n, 1)

x1 = u
x2 = 0.5 * u + v
x3 = 2.0 * u - 0.3 * v + 0.1 * torch.randn(n, 1)

X = torch.cat([x1, x2, x3], dim=1)

# Center the data
X = X - X.mean(dim=0, keepdim=True)
```

Each sample has shape:

$$
x \in \mathbb{R}^3
$$

The latent representation has shape:

$$
z \in \mathbb{R}^2
$$


---

## Requirements


1. Create a **linear autoencoder**:

```python
encoder: Linear(3, 2)
decoder: Linear(2, 3)
```

Do not use nonlinear activations in this task.


2. Train the autoencoder using:
   * `MSELoss`,
   * an optimizer from `torch.optim`,
   * at least 500 training steps.
3. Plot:
   * training loss vs iteration,
   * original 3D points,
   * reconstructed 3D points,
   * 2D latent codes produced by the encoder.
4. Fit PCA with 2 components to the same centered data.
5. Compare:
   * reconstruction MSE of the autoencoder,
   * reconstruction MSE of PCA.
6. Print a small table:

| Method | Latent dimension | Reconstruction MSE |
|--------|-----------------:|-------------------:|
| Linear AE | 2                | ...                |
| PCA    | 2                | ...                |


---

## Required discussion

Explain:

* what the encoder learns,
* what the decoder learns,
* why the bottleneck forces compression,
* why the result should be similar to PCA,
* why the latent coordinates of the autoencoder do not have to be identical to PCA coordinates,
* whether the autoencoder and PCA reached similar reconstruction errors.


---


# Task 2 — Denoising autoencoder on Fashion-MNIST (3 pts)

In this task, you will train an **==convolutional autoencoder==** that receives a **noisy image** and tries to reconstruct the **clean image**.

The model learns:

$$
\text{noisy image} \rightarrow \text{encoder} \rightarrow z \rightarrow \text{decoder} \rightarrow \text{clean image}
$$

This is called a **denoising autoencoder**.


---

## Dataset

Use Fashion-MNIST.

Each image has shape:

$$
x \in \mathbb{R}^{1 \times 28 \times 28}
$$

Use pixel values in the range $[0, 1]$.

To create noisy images, add Gaussian noise:

$$
\tilde{x} = x + \sigma \epsilon,
\qquad
\epsilon \sim \mathcal{N}(0, I)
$$

Then clamp the result to $[0, 1]$:

$$
\tilde{x} = \text{clamp}(\tilde{x}, 0, 1)
$$

Suggested noise level:

$$
\sigma = 0.3
$$


---

## Requirements


1. Load Fashion-MNIST using TorchVision.
2. Create a function `add_noise(x, noise_std)` that adds Gaussian noise and clamps the result.
3. Implement a **convolutional** **autoencoder**.
4. Train the model for at least 5 epochs.
5. Use:
   * noisy image as input,
   * clean image as target,
   * `MSELoss`.
6. Track and plot training loss.
7. Show at least 8 examples with three rows:
   * clean image,
   * noisy image,
   * reconstructed / denoised image.
8. Try at least two noise levels, for example:
   * $\sigma = 0.1$,
   * $\sigma = 0.3$,
   * optionally $\sigma = 0.5$.
9. Briefly compare the quality of denoising for different noise levels.


---

## Required discussion

Explain:

* why this is still an autoencoder,
* why the input and target are different,
* what the latent representation contains,
* why very large noise makes the task harder,
* whether the model removes noise or also blurs some details.


---

## Suggested model

You may use this architecture or create a similar one.

```python
class ConvDenoisingAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(
                # TODO
            ), # 28 -> 14
            nn.ReLU(),
            nn.Conv2d(
                # TODO
            ), # 14 -> 7
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                # TODO
            ), # 7 -> 14
            nn.ReLU(),
            nn.ConvTranspose2d(
                # TODO
            ), # 14 -> 28
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat
```


---

## Starter code

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()

full_train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

# You may use a subset if training is slow.
train_size = 50000
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
```

```python
def add_noise(x, noise_std=0.3):
    noise = # TODO
    x_noisy = x + noise
    x_noisy = torch.clamp(x_noisy, 0.0, 1.0)
    return x_noisy
```

```python
def train_one_epoch(model, loader, optimizer, criterion, device, noise_std):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for clean, _ in loader:
        clean = clean.to(device)
        noisy = add_noise(clean, noise_std=noise_std)

        optimizer.zero_grad()
        reconstructed = model(noisy)
        loss = criterion(reconstructed, clean)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * clean.size(0)
        total_samples += clean.size(0)

    return total_loss / total_samples
```




# Task 3 — First steps toward Variational Autoencoders (3 pts)

In this task, you do not need to train a full VAE. The goal is to understand the mathematical and implementation components that appear in the VAE loss.

A VAE encoder does not output one deterministic vector $z$. Instead, it outputs parameters of a distribution:

$$
q_\phi(z \mid x) = \mathcal{N}(\mu_\phi(x), \operatorname{diag}(\sigma_\phi^2(x)))
$$

Then we sample:

$$
z \sim q_\phi(z \mid x)
$$

and decode $z$.


---

## Part A — KL divergence between two 1D normal distributions

For two one-dimensional Gaussian distributions,

$$
q(z) = \mathcal{N}(\mu_q, \sigma_q^2)
$$

and

$$
p(z) = \mathcal{N}(\mu_p, \sigma_p^2),
$$

the KL divergence is:

$$
D_{KL}(q \| p)
=
\log \frac{\sigma_p}{\sigma_q}
+
\frac{\sigma_q^2 + (\mu_q - \mu_p)^2}{2\sigma_p^2}
-
\frac{1}{2}.
$$

Calculate the following by hand or using Python:


1. $q = \mathcal{N}(0, 1)$, $p = \mathcal{N}(0, 1)$
2. $q = \mathcal{N}(1, 1)$, $p = \mathcal{N}(0, 1)$
3. $q = \mathcal{N}(0, 0.5^2)$, $p = \mathcal{N}(0, 1)$
4. $q = \mathcal{N}(0, 2^2)$, $p = \mathcal{N}(0, 1)$

Create a table with columns:

| $\mu_q$ | $\sigma_q$ | $\mu_p$ | $\sigma_p$ | $D_{KL}(q \| p)$ |
|------:|---------:|------:|---------:|---------------:|


---

## Part B — KL term used in a standard VAE

In the most common VAE implementation, the prior is:

$$
p(z) = \mathcal{N}(0, I)
$$

and the encoder gives a diagonal Gaussian:

$$
q_\phi(z \mid x) = \mathcal{N}(\mu, \operatorname{diag}(\sigma^2)).
$$

The KL divergence is:

$$
D_{KL}(q_\phi(z \mid x) \| p(z))
=
\frac{1}{2}
\sum_{j=1}^{d}
\left(
\sigma_j^2 + \mu_j^2 - 1 - \log \sigma_j^2
\right).
$$

In code, we often use `logvar`:

$$
\log \text{var}_j = \log \sigma_j^2.
$$

Then:

$$
D_{KL}
=
-\frac{1}{2}
\sum_{j=1}^{d}
\left(
1 + \log \text{var}_j - \mu_j^2 - \exp(\log \text{var}_j)
\right).
$$

Implement a function:

```python
def kl_standard_normal(mu, logvar):
    """
    mu: tensor of shape (batch_size, latent_dim)
    logvar: tensor of shape (batch_size, latent_dim)
    returns: tensor of shape (batch_size,)
    """
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return kl
```

Test it on:

```python
mu = torch.tensor([[0.0, 0.0],
                   [1.0, 0.0],
                   [0.0, 2.0]])

logvar = torch.tensor([[0.0, 0.0],
                       [0.0, 0.0],
                       [0.0, 0.0]])
```

Explain the result.


---

## Part C — Reparameterization trick

The problem is that direct sampling,

$$
z \sim \mathcal{N}(\mu, \sigma^2),
$$

contains randomness. The reparameterization trick writes the same sample as:

$$
z = \mu + \sigma \odot \epsilon,
\qquad
\epsilon \sim \mathcal{N}(0, I).
$$

Implement:

```python
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + std * eps
    return z
```

Requirements:


1. Create tensors `mu` and `logvar`.
2. Generate at least 10000 samples for a simple 1D case.
3. Verify empirically that the sample mean is close to `mu`.
4. Verify empirically that the sample standard deviation is close to `sigma`.
5. Explain why this trick is useful for training neural networks with backpropagation.


---

## Optional / Bonus — Inference with a pretrained VAE from Hugging Face Diffusers (+2 pts)

This part is optional. TorchVision does not provide a standard pretrained VAE model, but Hugging Face Diffusers provides pretrained VAE modules such as `AutoencoderKL`.

You may use a pretrained VAE only for inference, not training.

Install if needed:

```bash
pip install diffusers accelerate transformers
```

Example:

```python
import torch
from diffusers import AutoencoderKL

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
vae = vae.to(device)
vae.eval()
```

Possible experiments:


1. Load one image, resize it, and convert it to a tensor in the range $[-1, 1]$.
2. Encode the image into a latent distribution.
3. Sample a latent vector from this distribution.
4. Decode the latent vector.
5. Visualize:
   * original image,
   * reconstructed image.
6. Try sampling random latent tensors with the same shape as the encoded latent, decode them using the VAE decoder, and discuss why the results are usually not semantically meaningful.
7. Discuss whether random decoded images are meaningful.

Important note: A Stable Diffusion VAE decoder alone is not the full Stable Diffusion model. Random latent vectors decoded by the VAE may not produce realistic images because the diffusion model is normally responsible for producing meaningful latent representations.


---

# ✅ Deliverables

Your notebook should contain:

* Task 1:
  * linear autoencoder implementation,
  * training curve,
  * PCA comparison,
  * reconstruction MSE table,
  * short discussion.
* Task 2:
  * denoising autoencoder implementation,
  * training curve,
  * denoising examples,
  * comparison for at least two noise levels,
  * short discussion.
* Task 3:
  * KL divergence calculations for 1D Gaussians,
  * implementation of VAE KL term for diagonal Gaussian posterior,
  * implementation of the reparameterization trick,
  * empirical sampling check,
  * short discussion.
* Optional:
  * pretrained VAE inference experiment,
  * original / reconstructed image visualization,
  * explanation of what the pretrained VAE can and cannot do.